# Calculate soil-gas flux

This notebook combines processed dC/dt NetCDF results with the auxiliary NetCDF files created by processing notebook 05.

It supports:

- Standard results with dimensions time, cutoff, and deadband.
- Best-Pareto MCMC results with dimensions time and MC.
- A single dC/dt file or every compatible file of one selected type in a folder.
- Optional environmental fallback from the closest selected donor chamber.
- First-sample or elapsed-time window-mean auxiliary reduction.
- Overlay previews for one or more calculated files.
- Preview-only y-range filtering and a time-based moving-window mean.

Source files are never modified. Export creates a new file whose name ends in _with_flux.nc. Preview filtering and smoothing never alter calculated or exported values.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import re
import sys
import urllib.parse

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from IPython.display import clear_output, display

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "soilgasflux_fcs").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "soilgasflux_fcs").exists():
    raise RuntimeError("Could not locate the repository root containing soilgasflux_fcs.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from soilgasflux_fcs.models import soilgasflux

STANDARD_SCHEMA = "standard"
MCMC_SCHEMA = "mcmc_best_pareto"
FLUX_SCHEMA_VERSION = "1.0"
DEFAULT_AUXILIARY_DIR = (
    PROJECT_ROOT / "notebooks" / "processing" / "output" / "auxiliary"
)
DEFAULT_FLUX_OUTPUT_DIR = (
    PROJECT_ROOT / "notebooks" / "post_processing" / "output" / "flux"
)

REQUIRED_MCMC_VARIABLES = {"dcdt(HM)", "best_deadband", "best_cutoff"}
REQUIRED_AUXILIARY_VARIABLES = {
    "pressure_kpa",
    "temperature_c",
    "relative_humidity_percent",
    "water_vapor_mmol_mol",
    "chamber_area_cm2",
    "chamber_volume_cm3",
    "pressure_quality_flag",
    "temperature_quality_flag",
    "relative_humidity_quality_flag",
    "water_vapor_quality_flag",
    "chamber_area_quality_flag",
    "chamber_volume_quality_flag",
    "measurement_quality_flag",
    "measurement_status",
    "measurement_status_detail",
}
REQUIRED_AUXILIARY_COORDS = {
    "timestamp",
    "elapsed_seconds",
    "measurement_id",
    "chamber_id",
}
ENVIRONMENT_FIELDS = [
    "pressure_kpa",
    "temperature_c",
    "relative_humidity_percent",
    "water_vapor_mmol_mol",
]
ENVIRONMENT_FLAG_FIELDS = [
    "pressure_quality_flag",
    "temperature_quality_flag",
    "relative_humidity_quality_flag",
    "water_vapor_quality_flag",
]


## dC/dt result discovery

The helpers below inspect file contents, classify supported result schemas, and determine the target chamber without relying only on a filename.


In [ ]:
def sanitize_path(path_value, label="Path"):
    text = str(path_value or "").strip()
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    if lines:
        text = lines[0]
    if not text:
        raise FileNotFoundError(f"{label} is empty.")

    if text.startswith(("Path(", "pathlib.Path(")) and text.endswith(")"):
        text = text[text.find("(") + 1:-1].strip()
    for _ in range(2):
        if len(text) >= 2 and text[0] == text[-1] and text[0] in {"'", '"'}:
            text = text[1:-1].strip()

    parsed = urllib.parse.urlparse(text)
    if parsed.scheme == "file":
        text = urllib.parse.unquote(parsed.path)
    if text.startswith("Users/"):
        text = "/" + text
    return Path(text).expanduser().resolve()


def utc_naive_timestamp(value):
    timestamp = pd.Timestamp(value)
    if pd.isna(timestamp):
        return pd.NaT
    if timestamp.tzinfo is not None:
        timestamp = timestamp.tz_convert("UTC").tz_localize(None)
    return timestamp


def utc_naive_index(values):
    index = pd.DatetimeIndex(pd.to_datetime(values))
    if index.tz is not None:
        index = index.tz_convert("UTC").tz_localize(None)
    return index


def classify_result_dataset(ds):
    dims = set(ds.sizes)
    variables = set(ds.data_vars)

    if "soil_gas_flux" in variables:
        return "already_with_flux"
    if (
        {"time", "cutoff", "deadband"}.issubset(dims)
        and "MC" not in dims
        and "dcdt(HM)" in variables
    ):
        return STANDARD_SCHEMA
    if (
        {"time", "MC"}.issubset(dims)
        and "cutoff" not in dims
        and "deadband" not in dims
        and REQUIRED_MCMC_VARIABLES.issubset(variables)
    ):
        return MCMC_SCHEMA
    if (
        {"time", "cutoff", "deadband", "MC"}.issubset(dims)
        and "dcdt(HM)" in variables
    ):
        return "unsupported_full_mcmc"
    return "unsupported"


def scalar_text(ds, name):
    if name in ds.coords or name in ds.data_vars:
        values = np.asarray(ds[name].values).reshape(-1)
        unique = pd.unique(values.astype(str))
        if len(unique) == 1 and unique[0].strip():
            return str(unique[0]).strip()
    value = ds.attrs.get(name)
    if value is not None and str(value).strip():
        return str(value).strip()
    return ""


def chamber_from_result(ds, path):
    chamber = scalar_text(ds, "chamber_id")
    if chamber:
        return chamber, "dataset chamber_id"

    stem = Path(path).stem
    match = re.match(
        r"^(?P<chamber>.+)_(?P<date>\d{4}-\d{2}-\d{2})(?:_.*)?$",
        stem,
    )
    if match and match.group("chamber").strip():
        return match.group("chamber").strip(), "filename"
    raise ValueError(
        "Could not determine chamber ID. Add a scalar chamber_id coordinate/attribute "
        "or use a package-style chamber_YYYY-MM-DD filename."
    )


def inspect_result_file(path):
    path = Path(path)
    record = {
        "path": path,
        "file": path.name,
        "schema": "unreadable",
        "chamber_id": "",
        "identity_source": "",
        "time_count": 0,
        "time_start": pd.NaT,
        "time_end": pd.NaT,
        "dimensions": "",
        "error": "",
    }
    if path.name.endswith("_with_flux.nc"):
        record["schema"] = "already_with_flux"
        return record

    try:
        with xr.open_dataset(path) as ds:
            schema = classify_result_dataset(ds)
            record["schema"] = schema
            record["dimensions"] = ", ".join(
                f"{name}={size}" for name, size in ds.sizes.items()
            )
            if schema in {STANDARD_SCHEMA, MCMC_SCHEMA}:
                chamber, source = chamber_from_result(ds, path)
                times = utc_naive_index(ds["time"].values)
                if times.isna().any():
                    raise ValueError("Result file contains invalid time values.")
                if times.duplicated().any():
                    raise ValueError("Result file contains duplicate time values.")
                record.update({
                    "chamber_id": chamber,
                    "identity_source": source,
                    "time_count": len(times),
                    "time_start": times.min(),
                    "time_end": times.max(),
                })
    except Exception as exc:
        record["schema"] = "unreadable"
        record["error"] = f"{type(exc).__name__}: {exc}"
    return record


def discover_result_inputs(mode, path_value, folder_schema=STANDARD_SCHEMA):
    path = sanitize_path(path_value, "dC/dt input path")
    if mode == "Single file":
        if not path.exists():
            raise FileNotFoundError(f"dC/dt file does not exist: {path}")
        if not path.is_file():
            raise IsADirectoryError(f"Single-file mode requires a file: {path}")
        record = inspect_result_file(path)
        if record["schema"] not in {STANDARD_SCHEMA, MCMC_SCHEMA}:
            detail = record["error"] or record["schema"]
            raise ValueError(f"{path.name} is not a supported dC/dt result: {detail}")
        record["selected"] = True
        return [record], [record]

    if not path.exists():
        raise FileNotFoundError(f"dC/dt folder does not exist: {path}")
    if not path.is_dir():
        raise NotADirectoryError(f"Folder mode requires a directory: {path}")

    paths = sorted(candidate for candidate in path.glob("*.nc") if candidate.is_file())
    if not paths:
        raise FileNotFoundError(f"No top-level NetCDF files were found in {path}.")

    records = [inspect_result_file(candidate) for candidate in paths]
    selected = []
    for record in records:
        record["selected"] = record["schema"] == folder_schema
        if record["selected"]:
            selected.append(record)
    if not selected:
        raise ValueError(
            f"No compatible {folder_schema} dC/dt files were found in {path}."
        )
    return selected, records


def result_records_table(records):
    return pd.DataFrame([
        {
            "selected": bool(record.get("selected", False)),
            "file": record["file"],
            "schema": record["schema"],
            "chamber_id": record["chamber_id"],
            "timestamps": record["time_count"],
            "start": record["time_start"],
            "end": record["time_end"],
            "dimensions": record["dimensions"],
            "diagnostic": record["error"],
        }
        for record in records
    ])


def load_result_dataset(path):
    with xr.open_dataset(path) as opened:
        return opened.load()


## Auxiliary catalog and matching

Auxiliary files are loaded explicitly and reduced into individual chamber measurements. Exact target matches provide geometry. Optional donors may replace the complete environmental tuple but never the target chamber geometry.


In [ ]:
def classify_auxiliary_dataset(ds):
    if "observation" not in ds.sizes:
        return False
    return (
        REQUIRED_AUXILIARY_VARIABLES.issubset(set(ds.data_vars))
        and REQUIRED_AUXILIARY_COORDS.issubset(set(ds.variables))
    )


def inspect_auxiliary_file(path):
    path = Path(path)
    record = {
        "path": path,
        "file": path.name,
        "schema": "unreadable",
        "chamber_id": "",
        "observations": 0,
        "measurements": 0,
        "time_start": pd.NaT,
        "time_end": pd.NaT,
        "error": "",
    }
    try:
        with xr.open_dataset(path) as ds:
            if not classify_auxiliary_dataset(ds):
                record["schema"] = "unsupported"
                return record
            chamber = scalar_text(ds, "chamber_id")
            if not chamber:
                raise ValueError("Auxiliary file has no scalar chamber_id.")
            timestamps = utc_naive_index(ds["timestamp"].values)
            measurement_ids = np.asarray(ds["measurement_id"].values).astype(str)
            record.update({
                "schema": "auxiliary",
                "chamber_id": chamber,
                "observations": int(ds.sizes["observation"]),
                "measurements": int(len(pd.unique(measurement_ids))),
                "time_start": timestamps.min(),
                "time_end": timestamps.max(),
            })
    except Exception as exc:
        record["schema"] = "unreadable"
        record["error"] = f"{type(exc).__name__}: {exc}"
    return record


def discover_auxiliary_files(folder_value):
    folder = sanitize_path(folder_value, "Auxiliary folder")
    if not folder.exists():
        raise FileNotFoundError(f"Auxiliary folder does not exist: {folder}")
    if not folder.is_dir():
        raise NotADirectoryError(f"Auxiliary path must be a folder: {folder}")
    paths = sorted(candidate for candidate in folder.glob("*.nc") if candidate.is_file())
    if not paths:
        raise FileNotFoundError(f"No top-level NetCDF files were found in {folder}.")
    records = [inspect_auxiliary_file(path) for path in paths]
    compatible = [record["path"] for record in records if record["schema"] == "auxiliary"]
    if not compatible:
        raise ValueError(f"No compatible auxiliary NetCDF files were found in {folder}.")
    return compatible, records


def _observation_values(ds, name, count):
    values = np.asarray(ds[name].values)
    if values.ndim == 0:
        return np.repeat(values.item(), count)
    values = values.reshape(-1)
    if len(values) != count:
        raise ValueError(
            f"Auxiliary variable {name!r} has {len(values)} values; expected {count}."
        )
    return values


def load_auxiliary_catalog(paths):
    observations = []
    for path in paths:
        path = Path(path)
        with xr.open_dataset(path) as opened:
            ds = opened.load()
        try:
            if not classify_auxiliary_dataset(ds):
                raise ValueError(f"{path.name} does not match the auxiliary schema.")
            count = int(ds.sizes["observation"])
            chamber = scalar_text(ds, "chamber_id")
            frame = pd.DataFrame({
                "timestamp": utc_naive_index(ds["timestamp"].values),
                "elapsed_seconds": _observation_values(ds, "elapsed_seconds", count).astype(float),
                "measurement_id": _observation_values(ds, "measurement_id", count).astype(str),
                "chamber_id": chamber,
                "source_file": str(path),
            })
            for name in REQUIRED_AUXILIARY_VARIABLES:
                frame[name] = _observation_values(ds, name, count)
            observations.append(frame)
        finally:
            ds.close()

    combined = pd.concat(observations, ignore_index=True)
    numeric_fields = (
        ENVIRONMENT_FIELDS
        + ENVIRONMENT_FLAG_FIELDS
        + [
            "chamber_area_cm2",
            "chamber_volume_cm3",
            "chamber_area_quality_flag",
            "chamber_volume_quality_flag",
            "measurement_quality_flag",
        ]
    )
    for field in numeric_fields:
        combined[field] = pd.to_numeric(combined[field], errors="coerce")
    combined = combined.sort_values(
        ["chamber_id", "timestamp", "measurement_id", "elapsed_seconds"]
    ).reset_index(drop=True)

    catalog = []
    group_columns = ["source_file", "chamber_id", "measurement_id"]
    for (source_file, chamber_id, measurement_id), group in combined.groupby(
        group_columns, sort=True, dropna=False
    ):
        group = group.sort_values(["elapsed_seconds", "timestamp"]).reset_index(drop=True)
        start = utc_naive_timestamp(group["timestamp"].min())
        catalog.append({
            "source_file": source_file,
            "chamber_id": str(chamber_id),
            "measurement_id": str(measurement_id),
            "measurement_start": start,
            "frame": group,
            "measurement_quality_flag": int(group["measurement_quality_flag"].iloc[0]),
            "measurement_status": str(group["measurement_status"].iloc[0]),
            "measurement_status_detail": str(group["measurement_status_detail"].iloc[0]),
        })
    return catalog


def auxiliary_records_table(records):
    return pd.DataFrame([
        {
            "file": record["file"],
            "schema": record["schema"],
            "chamber_id": record["chamber_id"],
            "observations": record["observations"],
            "measurements": record["measurements"],
            "start": record["time_start"],
            "end": record["time_end"],
            "diagnostic": record["error"],
        }
        for record in records
    ])


def exact_target_candidates(catalog, chamber_id, target_time):
    target_time = utc_naive_timestamp(target_time)
    return [
        measurement
        for measurement in catalog
        if measurement["chamber_id"] == str(chamber_id)
        and measurement["measurement_start"] == target_time
    ]


def reduce_geometry(measurement):
    first = measurement["frame"].sort_values(
        ["elapsed_seconds", "timestamp"]
    ).iloc[0]
    area = float(first["chamber_area_cm2"])
    volume = float(first["chamber_volume_cm3"])
    area_flag = int(first["chamber_area_quality_flag"])
    volume_flag = int(first["chamber_volume_quality_flag"])
    usable = (
        np.isfinite(area)
        and area > 0
        and np.isfinite(volume)
        and volume > 0
        and area_flag != 2
        and volume_flag != 2
    )
    quality = max(area_flag, volume_flag) if usable else 2
    detail = "" if usable else "Target chamber area or volume is missing or invalid."
    return {
        "usable": bool(usable),
        "chamber_area_cm2": area,
        "chamber_volume_cm3": volume,
        "quality": int(quality),
        "detail": detail,
    }


def reduce_environment(measurement, reduction_mode, deadband=None, cutoff=None):
    frame = measurement["frame"].sort_values(["elapsed_seconds", "timestamp"])
    if reduction_mode == "First sample":
        selected = frame.iloc[[0]]
    elif reduction_mode == "Window mean":
        if deadband is None or cutoff is None:
            return {"usable": False, "quality": 2, "detail": "Window bounds are unavailable."}
        deadband = float(deadband)
        cutoff = float(cutoff)
        if not np.isfinite(deadband) or not np.isfinite(cutoff) or cutoff <= deadband:
            return {"usable": False, "quality": 2, "detail": "Window bounds are invalid."}
        selected = frame[
            frame["elapsed_seconds"].ge(deadband)
            & frame["elapsed_seconds"].lt(cutoff)
        ]
    else:
        raise ValueError(f"Unsupported auxiliary reduction mode: {reduction_mode}")

    if selected.empty:
        return {
            "usable": False,
            "quality": 2,
            "detail": "No auxiliary observations fall inside the requested window.",
        }

    complete = np.ones(len(selected), dtype=bool)
    for value_field, flag_field in zip(ENVIRONMENT_FIELDS, ENVIRONMENT_FLAG_FIELDS):
        values = pd.to_numeric(selected[value_field], errors="coerce").to_numpy(dtype=float)
        flags = pd.to_numeric(selected[flag_field], errors="coerce").to_numpy(dtype=float)
        complete &= np.isfinite(values) & np.isfinite(flags) & (flags != 2)

    selected = selected.loc[complete]
    if selected.empty:
        return {
            "usable": False,
            "quality": 2,
            "detail": "Pressure, temperature, RH, or water vapor is unresolved.",
        }

    result = {
        field: float(pd.to_numeric(selected[field], errors="coerce").mean())
        for field in ENVIRONMENT_FIELDS
    }
    used_flags = np.column_stack([
        pd.to_numeric(selected[field], errors="coerce").to_numpy(dtype=float)
        for field in ENVIRONMENT_FLAG_FIELDS
    ])
    result.update({
        "usable": all(np.isfinite(result[field]) for field in ENVIRONMENT_FIELDS),
        "quality": int(1 if np.any(used_flags == 1) else 0),
        "detail": "",
    })
    if not result["usable"]:
        result["quality"] = 2
        result["detail"] = "Reduced environmental values are not finite."
    return result


def find_closest_donor(
    catalog,
    target_chamber,
    target_time,
    permitted_chambers,
    maximum_gap_minutes,
    reduction_mode,
    deadband=None,
    cutoff=None,
):
    target_time = utc_naive_timestamp(target_time)
    permitted = {str(value) for value in permitted_chambers}
    maximum_gap_seconds = float(maximum_gap_minutes) * 60.0
    if not np.isfinite(maximum_gap_seconds) or maximum_gap_seconds < 0:
        raise ValueError("Maximum donor gap must be finite and non-negative.")

    candidates = []
    for measurement in catalog:
        chamber = measurement["chamber_id"]
        if chamber == str(target_chamber) or chamber not in permitted:
            continue
        environment = reduce_environment(
            measurement, reduction_mode, deadband=deadband, cutoff=cutoff
        )
        if not environment["usable"]:
            continue
        signed_offset = (
            measurement["measurement_start"] - target_time
        ).total_seconds()
        absolute_offset = abs(signed_offset)
        if absolute_offset <= maximum_gap_seconds:
            candidates.append((
                absolute_offset,
                chamber,
                measurement["measurement_id"],
                measurement,
                environment,
                signed_offset,
            ))

    if not candidates:
        return None
    candidates.sort(key=lambda item: (item[0], item[1], item[2]))
    _, _, _, measurement, environment, signed_offset = candidates[0]
    return measurement, environment, float(signed_offset)


def unresolved_resolution(status, detail, target_measurement_id=""):
    return {
        "usable": False,
        "quality": 2,
        "substitution_flag": 2,
        "status": status,
        "detail": detail,
        "target_measurement_id": str(target_measurement_id),
        "source_measurement_id": "",
        "source_chamber_id": "",
        "source_timestamp": pd.NaT,
        "time_offset_seconds": np.nan,
        "pressure_kpa": np.nan,
        "temperature_c": np.nan,
        "relative_humidity_percent": np.nan,
        "water_vapor_mmol_mol": np.nan,
        "chamber_area_cm2": np.nan,
        "chamber_volume_cm3": np.nan,
    }


def resolve_auxiliary_inputs(
    catalog,
    target_chamber,
    target_time,
    reduction_mode,
    use_other_chambers=False,
    permitted_donor_chambers=(),
    maximum_gap_minutes=30.0,
    deadband=None,
    cutoff=None,
):
    matches = exact_target_candidates(catalog, target_chamber, target_time)
    if not matches:
        return unresolved_resolution(
            "missing_target_auxiliary",
            "No auxiliary measurement exactly matches the target chamber and timestamp.",
        )
    if len(matches) > 1:
        return unresolved_resolution(
            "ambiguous_target_auxiliary",
            f"{len(matches)} auxiliary measurements exactly match the target chamber and timestamp.",
        )

    target = matches[0]
    geometry = reduce_geometry(target)
    if not geometry["usable"]:
        return unresolved_resolution(
            "invalid_target_geometry",
            geometry["detail"],
            target["measurement_id"],
        )

    environment = reduce_environment(
        target, reduction_mode, deadband=deadband, cutoff=cutoff
    )
    if environment["usable"]:
        quality = max(
            int(environment["quality"]),
            int(geometry["quality"]),
            min(max(int(target["measurement_quality_flag"]), 0), 2),
        )
        quality = 0 if quality == 0 else 1
        status = "direct_original" if quality == 0 else "direct_filled_or_default"
        detail = (
            "Used the exact target auxiliary measurement."
            if quality == 0
            else "Used the exact target measurement with interpolation, edge filling, or geometry defaults."
        )
        return {
            "usable": True,
            "quality": quality,
            "substitution_flag": 0,
            "status": status,
            "detail": detail,
            "target_measurement_id": target["measurement_id"],
            "source_measurement_id": target["measurement_id"],
            "source_chamber_id": target["chamber_id"],
            "source_timestamp": target["measurement_start"],
            "time_offset_seconds": 0.0,
            **{field: environment[field] for field in ENVIRONMENT_FIELDS},
            "chamber_area_cm2": geometry["chamber_area_cm2"],
            "chamber_volume_cm3": geometry["chamber_volume_cm3"],
        }

    if not use_other_chambers:
        return unresolved_resolution(
            "invalid_target_environment",
            environment["detail"] + " Other-chamber fallback is disabled.",
            target["measurement_id"],
        )

    donor = find_closest_donor(
        catalog,
        target_chamber,
        target_time,
        permitted_donor_chambers,
        maximum_gap_minutes,
        reduction_mode,
        deadband=deadband,
        cutoff=cutoff,
    )
    if donor is None:
        return unresolved_resolution(
            "no_acceptable_donor",
            "No selected donor chamber has a complete environmental tuple within the maximum gap.",
            target["measurement_id"],
        )

    donor_measurement, donor_environment, signed_offset = donor
    return {
        "usable": True,
        "quality": 1,
        "substitution_flag": 1,
        "status": "substituted_other_chamber",
        "detail": (
            f"Environmental values came from chamber {donor_measurement['chamber_id']} "
            f"at a signed offset of {signed_offset:.1f} seconds; target geometry was retained."
        ),
        "target_measurement_id": target["measurement_id"],
        "source_measurement_id": donor_measurement["measurement_id"],
        "source_chamber_id": donor_measurement["chamber_id"],
        "source_timestamp": donor_measurement["measurement_start"],
        "time_offset_seconds": signed_offset,
        **{field: donor_environment[field] for field in ENVIRONMENT_FIELDS},
        "chamber_area_cm2": geometry["chamber_area_cm2"],
        "chamber_volume_cm3": geometry["chamber_volume_cm3"],
    }


def summarize_primary_matches(result_records, catalog):
    rows = []
    for record in result_records:
        ds = load_result_dataset(record["path"])
        try:
            for target_time in utc_naive_index(ds["time"].values):
                matches = exact_target_candidates(
                    catalog, record["chamber_id"], target_time
                )
                if not matches:
                    state = "missing"
                    measurement_id = ""
                    measurement_status = ""
                    detail = "No exact chamber/timestamp match."
                elif len(matches) > 1:
                    state = "ambiguous"
                    measurement_id = ""
                    measurement_status = ""
                    detail = f"{len(matches)} exact matches."
                else:
                    target = matches[0]
                    geometry = reduce_geometry(target)
                    environment = reduce_environment(target, "First sample")
                    state = "matched_valid" if geometry["usable"] and environment["usable"] else "matched_invalid"
                    measurement_id = target["measurement_id"]
                    measurement_status = target["measurement_status"]
                    details = [value for value in [geometry["detail"], environment["detail"]] if value]
                    detail = "; ".join(details) or target["measurement_status_detail"]
                rows.append({
                    "result_file": record["file"],
                    "schema": record["schema"],
                    "target_chamber": record["chamber_id"],
                    "dcdt_time": target_time,
                    "match_state": state,
                    "target_measurement_id": measurement_id,
                    "target_measurement_status": measurement_status,
                    "detail": detail,
                })
        finally:
            ds.close()
    return pd.DataFrame(rows)


## Flux conversion and export

Standard files retain their complete grid. Best-Pareto MCMC files retain every posterior sample. Invalid or unmatched cells remain present with NaN flux and explicit provenance.


In [ ]:
STANDARD_METADATA_FIELDS = [
    "flux_pressure_kpa",
    "flux_temperature_c",
    "flux_relative_humidity_percent",
    "flux_water_vapor_mmol_mol",
    "flux_chamber_area_cm2",
    "flux_chamber_volume_cm3",
    "flux_quality_flag",
    "auxiliary_substitution_flag",
    "auxiliary_target_measurement_id",
    "auxiliary_source_measurement_id",
    "auxiliary_source_chamber_id",
    "auxiliary_source_timestamp",
    "auxiliary_time_offset_seconds",
    "flux_status",
    "flux_status_detail",
]


def flux_from_resolution(dcdt_values, resolution):
    if not resolution["usable"]:
        return np.full_like(np.asarray(dcdt_values, dtype=float), np.nan, dtype=float)
    values = soilgasflux(
        volume=resolution["chamber_volume_cm3"],
        area=resolution["chamber_area_cm2"],
        p0=resolution["pressure_kpa"],
        w0=resolution["water_vapor_mmol_mol"],
        t0=resolution["temperature_c"],
        dcdt=np.asarray(dcdt_values, dtype=float),
    )
    values = np.asarray(values, dtype=float)
    return np.where(np.isfinite(values), values, np.nan)


def empty_calculation_arrays(shape):
    arrays = {
        "soil_gas_flux": np.full(shape, np.nan, dtype=float),
        "flux_pressure_kpa": np.full(shape, np.nan, dtype=float),
        "flux_temperature_c": np.full(shape, np.nan, dtype=float),
        "flux_relative_humidity_percent": np.full(shape, np.nan, dtype=float),
        "flux_water_vapor_mmol_mol": np.full(shape, np.nan, dtype=float),
        "flux_chamber_area_cm2": np.full(shape, np.nan, dtype=float),
        "flux_chamber_volume_cm3": np.full(shape, np.nan, dtype=float),
        "flux_quality_flag": np.full(shape, 2, dtype=np.int8),
        "auxiliary_substitution_flag": np.full(shape, 2, dtype=np.int8),
        "auxiliary_target_measurement_id": np.full(shape, "", dtype="<U256"),
        "auxiliary_source_measurement_id": np.full(shape, "", dtype="<U256"),
        "auxiliary_source_chamber_id": np.full(shape, "", dtype="<U256"),
        "auxiliary_source_timestamp": np.full(
            shape, np.datetime64("NaT"), dtype="datetime64[ns]"
        ),
        "auxiliary_time_offset_seconds": np.full(shape, np.nan, dtype=float),
        "flux_status": np.full(shape, "invalid_dcdt", dtype="<U256"),
        "flux_status_detail": np.full(
            shape, "dcdt(HM) is not finite.", dtype="<U512"
        ),
    }
    return arrays


def store_resolution(arrays, index, resolution):
    mapping = {
        "flux_pressure_kpa": "pressure_kpa",
        "flux_temperature_c": "temperature_c",
        "flux_relative_humidity_percent": "relative_humidity_percent",
        "flux_water_vapor_mmol_mol": "water_vapor_mmol_mol",
        "flux_chamber_area_cm2": "chamber_area_cm2",
        "flux_chamber_volume_cm3": "chamber_volume_cm3",
        "flux_quality_flag": "quality",
        "auxiliary_substitution_flag": "substitution_flag",
        "auxiliary_target_measurement_id": "target_measurement_id",
        "auxiliary_source_measurement_id": "source_measurement_id",
        "auxiliary_source_chamber_id": "source_chamber_id",
        "auxiliary_time_offset_seconds": "time_offset_seconds",
        "flux_status": "status",
        "flux_status_detail": "detail",
    }
    for output_name, resolution_name in mapping.items():
        arrays[output_name][index] = resolution[resolution_name]
    timestamp = resolution["source_timestamp"]
    arrays["auxiliary_source_timestamp"][index] = (
        np.datetime64(timestamp.to_datetime64())
        if isinstance(timestamp, pd.Timestamp) and not pd.isna(timestamp)
        else np.datetime64("NaT")
    )


def data_array_from_canonical(values, canonical_dims, canonical_coords, target_dims):
    array = xr.DataArray(values, dims=canonical_dims, coords=canonical_coords)
    return array.transpose(*target_dims)


def add_flux_variable_attributes(ds):
    attrs = {
        "soil_gas_flux": {
            "long_name": "soil gas flux calculated from Hutchinson-Mosier dC/dt",
            "units": "umol m-2 s-1",
        },
        "flux_pressure_kpa": {"long_name": "pressure used for flux conversion", "units": "kPa"},
        "flux_temperature_c": {"long_name": "temperature used for flux conversion", "units": "degree_Celsius"},
        "flux_relative_humidity_percent": {
            "long_name": "relative humidity associated with the flux conversion",
            "units": "percent",
        },
        "flux_water_vapor_mmol_mol": {
            "long_name": "water vapor mole fraction used for flux conversion",
            "units": "mmol mol-1",
        },
        "flux_chamber_area_cm2": {"long_name": "target chamber area used for flux conversion", "units": "cm2"},
        "flux_chamber_volume_cm3": {"long_name": "target chamber volume used for flux conversion", "units": "cm3"},
        "flux_quality_flag": {
            "flag_values": np.array([0, 1, 2], dtype=np.int8),
            "flag_meanings": "direct_original direct_filled_or_default_or_other_chamber invalid_unresolved",
        },
        "auxiliary_substitution_flag": {
            "flag_values": np.array([0, 1, 2], dtype=np.int8),
            "flag_meanings": "target_chamber other_chamber invalid_unresolved",
        },
        "auxiliary_time_offset_seconds": {
            "long_name": "signed source auxiliary start time minus target dC/dt time",
            "units": "s",
        },
    }
    for name, variable_attrs in attrs.items():
        ds[name].attrs.update(variable_attrs)


def apply_flux_global_attributes(
    ds,
    source_path,
    auxiliary_paths,
    target_chamber,
    reduction_mode,
    use_other_chambers,
    permitted_donor_chambers,
    maximum_gap_minutes,
):
    ds.attrs.update({
        "flux_schema_version": FLUX_SCHEMA_VERSION,
        "flux_created_utc": datetime.now(timezone.utc).isoformat(),
        "flux_formula": (
            "F = 10 * volume_cm3 * pressure_kPa * (1 - water_vapor_mmol_mol / 1000) "
            "* dcdt_ppm_s / (R * area_cm2 * (temperature_C + 273.15))"
        ),
        "flux_source_dcdt_file": str(Path(source_path).resolve()),
        "flux_source_auxiliary_files": json.dumps(
            sorted(str(Path(path).resolve()) for path in auxiliary_paths)
        ),
        "flux_target_chamber_id": str(target_chamber),
        "flux_auxiliary_reduction": str(reduction_mode),
        "flux_exact_target_timestamp_match": "true",
        "flux_other_chamber_fallback_enabled": str(bool(use_other_chambers)).lower(),
        "flux_permitted_donor_chambers": json.dumps(
            sorted(str(value) for value in permitted_donor_chambers)
        ),
        "flux_maximum_donor_gap_minutes": float(maximum_gap_minutes),
        "flux_quality_flag_definition": (
            "0=direct_original, "
            "1=direct_filled_or_default_or_other_chamber, "
            "2=invalid_unresolved"
        ),
        "auxiliary_substitution_flag_definition": (
            "0=target_chamber, 1=other_chamber, 2=invalid_unresolved"
        ),
    })


def calculate_standard_flux(
    source_ds,
    source_path,
    target_chamber,
    catalog,
    auxiliary_paths,
    reduction_mode,
    use_other_chambers,
    permitted_donor_chambers,
    maximum_gap_minutes,
):
    dcdt = source_ds["dcdt(HM)"].transpose("time", "cutoff", "deadband")
    values = np.asarray(dcdt.values, dtype=float)
    arrays = empty_calculation_arrays(values.shape)
    times = utc_naive_index(dcdt["time"].values)
    cutoffs = np.asarray(dcdt["cutoff"].values)
    deadbands = np.asarray(dcdt["deadband"].values)

    for time_index, target_time in enumerate(times):
        for cutoff_index, cutoff in enumerate(cutoffs):
            for deadband_index, deadband in enumerate(deadbands):
                index = (time_index, cutoff_index, deadband_index)
                dcdt_value = values[index]
                if not np.isfinite(dcdt_value):
                    continue
                resolution = resolve_auxiliary_inputs(
                    catalog,
                    target_chamber,
                    target_time,
                    reduction_mode,
                    use_other_chambers=use_other_chambers,
                    permitted_donor_chambers=permitted_donor_chambers,
                    maximum_gap_minutes=maximum_gap_minutes,
                    deadband=deadband,
                    cutoff=cutoff,
                )
                store_resolution(arrays, index, resolution)
                arrays["soil_gas_flux"][index] = flux_from_resolution(
                    dcdt_value, resolution
                )

    output = source_ds.copy(deep=True)
    canonical_dims = ("time", "cutoff", "deadband")
    canonical_coords = {
        "time": source_ds["time"],
        "cutoff": source_ds["cutoff"],
        "deadband": source_ds["deadband"],
    }
    target_dims = source_ds["dcdt(HM)"].dims
    for name, values_array in arrays.items():
        output[name] = data_array_from_canonical(
            values_array, canonical_dims, canonical_coords, target_dims
        )
    add_flux_variable_attributes(output)
    apply_flux_global_attributes(
        output, source_path, auxiliary_paths, target_chamber, reduction_mode,
        use_other_chambers, permitted_donor_chambers, maximum_gap_minutes,
    )
    return output


def calculate_mcmc_flux(
    source_ds,
    source_path,
    target_chamber,
    catalog,
    auxiliary_paths,
    reduction_mode,
    use_other_chambers,
    permitted_donor_chambers,
    maximum_gap_minutes,
):
    dcdt = source_ds["dcdt(HM)"].transpose("time", "MC")
    values = np.asarray(dcdt.values, dtype=float)
    flux = np.full(values.shape, np.nan, dtype=float)
    metadata = empty_calculation_arrays((values.shape[0],))
    times = utc_naive_index(dcdt["time"].values)

    for time_index, target_time in enumerate(times):
        finite = np.isfinite(values[time_index])
        if not finite.any():
            continue
        deadband = float(source_ds["best_deadband"].isel(time=time_index).values)
        cutoff = float(source_ds["best_cutoff"].isel(time=time_index).values)
        resolution = resolve_auxiliary_inputs(
            catalog,
            target_chamber,
            target_time,
            reduction_mode,
            use_other_chambers=use_other_chambers,
            permitted_donor_chambers=permitted_donor_chambers,
            maximum_gap_minutes=maximum_gap_minutes,
            deadband=deadband,
            cutoff=cutoff,
        )
        store_resolution(metadata, time_index, resolution)
        flux[time_index] = flux_from_resolution(values[time_index], resolution)

    output = source_ds.copy(deep=True)
    canonical_flux = xr.DataArray(
        flux,
        dims=("time", "MC"),
        coords={"time": source_ds["time"], "MC": source_ds["MC"]},
    ).transpose(*source_ds["dcdt(HM)"].dims)
    output["soil_gas_flux"] = canonical_flux
    for name, values_array in metadata.items():
        if name == "soil_gas_flux":
            continue
        output[name] = xr.DataArray(
            values_array, dims=("time",), coords={"time": source_ds["time"]}
        )
    add_flux_variable_attributes(output)
    apply_flux_global_attributes(
        output, source_path, auxiliary_paths, target_chamber, reduction_mode,
        use_other_chambers, permitted_donor_chambers, maximum_gap_minutes,
    )
    return output


def calculate_flux_for_record(
    record,
    catalog,
    auxiliary_paths,
    reduction_mode,
    use_other_chambers=False,
    permitted_donor_chambers=(),
    maximum_gap_minutes=30.0,
):
    source = load_result_dataset(record["path"])
    try:
        schema = classify_result_dataset(source)
        if schema != record["schema"]:
            raise ValueError(
                f"{record['file']} changed schema between scanning and calculation."
            )
        if schema == STANDARD_SCHEMA:
            output = calculate_standard_flux(
                source, record["path"], record["chamber_id"], catalog,
                auxiliary_paths, reduction_mode, use_other_chambers,
                permitted_donor_chambers, maximum_gap_minutes,
            )
        elif schema == MCMC_SCHEMA:
            output = calculate_mcmc_flux(
                source, record["path"], record["chamber_id"], catalog,
                auxiliary_paths, reduction_mode, use_other_chambers,
                permitted_donor_chambers, maximum_gap_minutes,
            )
        else:
            raise ValueError(f"Unsupported result schema: {schema}")
    finally:
        source.close()
    return {
        "source_path": Path(record["path"]),
        "file": record["file"],
        "schema": record["schema"],
        "chamber_id": record["chamber_id"],
        "dataset": output,
    }


def calculate_flux_batch(
    records,
    catalog,
    auxiliary_paths,
    reduction_mode,
    use_other_chambers=False,
    permitted_donor_chambers=(),
    maximum_gap_minutes=30.0,
):
    calculated = []
    errors = []
    for record in records:
        try:
            calculated.append(calculate_flux_for_record(
                record,
                catalog,
                auxiliary_paths,
                reduction_mode,
                use_other_chambers=use_other_chambers,
                permitted_donor_chambers=permitted_donor_chambers,
                maximum_gap_minutes=maximum_gap_minutes,
            ))
        except Exception as exc:
            errors.append({
                "file": record["file"],
                "error": f"{type(exc).__name__}: {exc}",
            })
    return calculated, errors


def flux_result_counts(entries):
    rows = []
    for entry in entries:
        ds = entry["dataset"]
        flags = np.asarray(ds["flux_quality_flag"].values)
        rows.append({
            "file": entry["file"],
            "schema": entry["schema"],
            "chamber_id": entry["chamber_id"],
            "valid_original": int((flags == 0).sum()),
            "usable_filled_or_donor": int((flags == 1).sum()),
            "invalid_unresolved": int((flags == 2).sum()),
            "finite_flux_values": int(np.isfinite(ds["soil_gas_flux"].values).sum()),
        })
    return pd.DataFrame(rows)


def export_flux_entries(entries, output_folder, overwrite=False):
    if not entries:
        raise ValueError("Calculate flux before exporting.")
    output_folder = sanitize_path(output_folder, "Flux output folder")
    targets = [
        output_folder / f"{entry['source_path'].stem}_with_flux.nc"
        for entry in entries
    ]
    existing = [path for path in targets if path.exists()]
    if existing and not overwrite:
        examples = ", ".join(path.name for path in existing[:5])
        raise FileExistsError(
            f"{len(existing)} output file(s) already exist. Enable overwrite first. "
            f"Examples: {examples}"
        )

    output_folder.mkdir(parents=True, exist_ok=True)
    written = []
    for entry, target in zip(entries, targets):
        entry["dataset"].to_netcdf(target)
        written.append(target)
    return written


## Preview helpers

The selected result can be inspected as a Standard time series or as an MCMC posterior median and 16–84% interval. Provenance is shown for the same selection.


In [ ]:
def finite_bounds(values):
    array = np.asarray(values, dtype=float).ravel()
    finite = array[np.isfinite(array)]
    if not finite.size:
        raise ValueError("No finite flux values are available for the y-range slider.")
    lower = float(finite.min())
    upper = float(finite.max())
    if np.isclose(lower, upper):
        padding = max(abs(lower) * 0.10, 0.01)
    else:
        padding = 0.05 * (upper - lower)
    slider_min = lower - padding
    slider_max = upper + padding
    step = max((slider_max - slider_min) / 200.0, np.finfo(float).eps)
    return slider_min, slider_max, step


def timestamp_options(values):
    index = utc_naive_index(values).dropna().unique().sort_values()
    return [
        (timestamp.strftime("%Y-%m-%d %H:%M:%S"), timestamp.to_datetime64())
        for timestamp in index
    ]


def selected_flux_view(entry, deadband=None, cutoff=None):
    ds = entry["dataset"]
    if entry["schema"] == STANDARD_SCHEMA:
        if deadband is None or cutoff is None:
            raise ValueError("Select a deadband and cutoff for the Standard preview.")
        flux = ds["soil_gas_flux"].sel(deadband=deadband, cutoff=cutoff)
        return pd.DataFrame(
            {"flux": np.asarray(flux.values, dtype=float)},
            index=utc_naive_index(ds["time"].values),
        )
    flux = ds["soil_gas_flux"]
    return pd.DataFrame(
        {
            "q16": np.asarray(flux.quantile(0.16, dim="MC", skipna=True).values, dtype=float),
            "median": np.asarray(flux.median(dim="MC", skipna=True).values, dtype=float),
            "q84": np.asarray(flux.quantile(0.84, dim="MC", skipna=True).values, dtype=float),
        },
        index=utc_naive_index(ds["time"].values),
    )


def provenance_table(entry, deadband=None, cutoff=None):
    ds = entry["dataset"]
    names = [
        "flux_quality_flag",
        "auxiliary_substitution_flag",
        "auxiliary_target_measurement_id",
        "auxiliary_source_measurement_id",
        "auxiliary_source_chamber_id",
        "auxiliary_source_timestamp",
        "auxiliary_time_offset_seconds",
        "flux_status",
        "flux_status_detail",
        "flux_pressure_kpa",
        "flux_temperature_c",
        "flux_water_vapor_mmol_mol",
        "flux_chamber_area_cm2",
        "flux_chamber_volume_cm3",
    ]
    columns = {}
    for name in names:
        data = ds[name]
        if entry["schema"] == STANDARD_SCHEMA:
            data = data.sel(deadband=deadband, cutoff=cutoff)
        columns[name] = np.asarray(data.values)
    frame = pd.DataFrame(columns, index=utc_naive_index(ds["time"].values))
    frame.index.name = "dcdt_time"
    return frame.reset_index()


def smoothing_offset(window_size, window_unit):
    size = int(window_size)
    if size < 1:
        raise ValueError("Moving-window size must be at least 1.")
    aliases = {"minutes": "min", "hours": "h", "days": "d"}
    if window_unit not in aliases:
        raise ValueError(f"Unsupported moving-window unit: {window_unit!r}")
    return pd.Timedelta(size, unit=aliases[window_unit])


def filter_flux_view(view, schema, y_range, enabled=True):
    filtered = view.copy()
    if not enabled or not y_range:
        return filtered
    lower, upper = map(float, y_range)
    if schema == STANDARD_SCHEMA:
        filtered["flux"] = filtered["flux"].where(
            filtered["flux"].between(lower, upper, inclusive="both")
        )
    else:
        keep = filtered["median"].between(lower, upper, inclusive="both")
        filtered.loc[~keep, ["q16", "median", "q84"]] = np.nan
    return filtered


def smooth_flux_view(
    view, schema, enabled=False, window_size=60, window_unit="minutes"
):
    if not enabled:
        return view.copy()
    offset = smoothing_offset(window_size, window_unit)
    smoothed = view.copy()
    columns = ["flux"] if schema == STANDARD_SCHEMA else ["q16", "median", "q84"]
    smoothed[columns] = view[columns].sort_index().rolling(
        offset, center=True, min_periods=1
    ).mean()
    return smoothed


def count_flux_outside_y_range(view, schema, y_range):
    if not y_range:
        return 0
    values = view["flux"] if schema == STANDARD_SCHEMA else view["median"]
    array = np.asarray(values, dtype=float)
    finite = np.isfinite(array)
    lower, upper = map(float, y_range)
    return int((finite & ((array < lower) | (array > upper))).sum())


def plot_flux_entry(
    entry,
    deadband,
    cutoff,
    date_range,
    y_range,
    filter_y=True,
    smooth=False,
    window_size=60,
    window_unit="minutes",
):
    view = selected_flux_view(entry, deadband=deadband, cutoff=cutoff)
    if date_range:
        start, end = [utc_naive_timestamp(value) for value in date_range]
        view = view.loc[(view.index >= start) & (view.index <= end)]
    view = filter_flux_view(view, entry["schema"], y_range, enabled=filter_y)
    view = smooth_flux_view(
        view, entry["schema"], enabled=smooth,
        window_size=window_size, window_unit=window_unit,
    )
    view = filter_flux_view(view, entry["schema"], y_range, enabled=filter_y)

    finite_columns = ["flux"] if entry["schema"] == STANDARD_SCHEMA else ["q16", "median", "q84"]
    if not np.isfinite(view[finite_columns].to_numpy(dtype=float)).any():
        raise ValueError(
            "No finite flux values remain inside the selected timestamp and y ranges."
        )

    smooth_label = (
        f", {window_size} {window_unit} moving mean" if smooth else ""
    )
    fig, ax = plt.subplots(figsize=(10, 4), dpi=110)
    if entry["schema"] == STANDARD_SCHEMA:
        ax.plot(
            view.index, view["flux"], "-o", color="#1769aa",
            linewidth=1.2, markersize=3,
            label=f"deadband={deadband}, cutoff={cutoff}{smooth_label}",
        )
    else:
        ax.fill_between(
            view.index, view["q16"], view["q84"],
            color="#1769aa", alpha=0.22, label="16–84%",
        )
        ax.plot(
            view.index, view["median"], "-o", color="#1769aa",
            linewidth=1.2, markersize=3,
            label=f"posterior median{smooth_label}",
        )
    ax.axhline(0.0, color="0.35", linewidth=0.8, alpha=0.5)
    ax.set_xlabel("Measurement start time [UTC]")
    ax.set_ylabel("Soil-gas flux [umol m-2 s-1]")
    ax.set_title(f"{entry['chamber_id']} — {entry['file']}")
    ax.grid(alpha=0.25)
    ax.legend(loc="best")
    if y_range:
        ax.set_ylim(*map(float, y_range))
    fig.autofmt_xdate()
    fig.tight_layout()
    return fig


def plot_flux_entries(
    entries,
    deadband,
    cutoff,
    date_range,
    y_range,
    filter_y=True,
    smooth=False,
    window_size=60,
    window_unit="minutes",
):
    entries = list(entries)
    if not entries:
        raise ValueError("Select at least one calculated result to preview.")
    schemas = {entry["schema"] for entry in entries}
    if len(schemas) != 1:
        raise ValueError("Selected previews must all use the same result schema.")
    schema = entries[0]["schema"]
    colors = plt.get_cmap("tab10")
    fig, ax = plt.subplots(figsize=(11.5, 4.5), dpi=110)
    plotted = 0

    for number, entry in enumerate(entries):
        view = selected_flux_view(entry, deadband=deadband, cutoff=cutoff)
        if date_range:
            start, end = [utc_naive_timestamp(value) for value in date_range]
            view = view.loc[(view.index >= start) & (view.index <= end)]
        view = filter_flux_view(view, schema, y_range, enabled=filter_y)
        view = smooth_flux_view(
            view, schema, enabled=smooth,
            window_size=window_size, window_unit=window_unit,
        )
        view = filter_flux_view(view, schema, y_range, enabled=filter_y)
        columns = ["flux"] if schema == STANDARD_SCHEMA else ["q16", "median", "q84"]
        finite = np.isfinite(view[columns].to_numpy(dtype=float))
        if not finite.any():
            continue

        color = colors(number % 10)
        label = f"{entry['chamber_id']} — {entry['file']}"
        smooth_label = (
            f" ({window_size} {window_unit} moving mean)" if smooth else ""
        )
        if schema == STANDARD_SCHEMA:
            ax.plot(
                view.index, view["flux"], "-o", color=color,
                linewidth=1.2, markersize=3, label=label + smooth_label,
            )
        else:
            ax.fill_between(
                view.index, view["q16"], view["q84"],
                color=color, alpha=0.16, label=label + " 16–84%",
            )
            ax.plot(
                view.index, view["median"], "-o", color=color,
                linewidth=1.2, markersize=3,
                label=label + " median" + smooth_label,
            )
        plotted += 1

    if not plotted:
        plt.close(fig)
        raise ValueError(
            "No selected file has finite flux inside the timestamp and y ranges."
        )
    ax.axhline(0.0, color="0.35", linewidth=0.8, alpha=0.5)
    ax.set_xlabel("Measurement start time [UTC]")
    ax.set_ylabel("Soil-gas flux [umol m-2 s-1]")
    title = f"Flux comparison — {plotted} file(s)"
    if schema == STANDARD_SCHEMA:
        title += f" — deadband={deadband}, cutoff={cutoff}"
    ax.set_title(title)
    ax.grid(alpha=0.25)
    ax.legend(loc="best", fontsize=8)
    if y_range:
        ax.set_ylim(*map(float, y_range))
    fig.autofmt_xdate()
    fig.tight_layout()
    return fig


## Interactive workflow

Use the tabs from left to right. Result files are selected first, auxiliary files automatically follow by exact chamber/timestamp matching, and calculated files are exported only after review.


In [ ]:
state = {
    "updating": False,
    "result_records": [],
    "result_scan_records": [],
    "auxiliary_paths": [],
    "auxiliary_records": [],
    "auxiliary_catalog": [],
    "calculated": [],
}

style = {"description_width": "175px"}
medium = widgets.Layout(width="430px")
wide = widgets.Layout(width="860px")
path_layout = widgets.Layout(width="860px", height="70px")

result_mode_widget = widgets.ToggleButtons(
    options=["Single file", "Whole folder"],
    value="Single file",
    description="Result input",
    style=style,
)
result_path_widget = widgets.Textarea(
    value="",
    description="dC/dt path",
    placeholder="Paste one NetCDF file or a folder path",
    continuous_update=False,
    style=style,
    layout=path_layout,
)
folder_schema_widget = widgets.Dropdown(
    options=[("Standard", STANDARD_SCHEMA), ("Best-Pareto MCMC", MCMC_SCHEMA)],
    value=STANDARD_SCHEMA,
    description="Folder result type",
    disabled=True,
    style=style,
    layout=medium,
)
scan_results_button = widgets.Button(
    description="Scan dC/dt input", button_style="primary"
)
result_status_widget = widgets.HTML(value="No dC/dt input selected.")
result_output = widgets.Output()

auxiliary_path_widget = widgets.Textarea(
    value=str(DEFAULT_AUXILIARY_DIR),
    description="Auxiliary folder",
    continuous_update=False,
    style=style,
    layout=path_layout,
)
scan_auxiliary_button = widgets.Button(
    description="Scan auxiliary folder", button_style="primary", disabled=True
)
reduction_widget = widgets.Dropdown(
    options=["First sample", "Window mean"],
    value="First sample",
    description="Auxiliary reduction",
    style=style,
    layout=medium,
)
use_donor_widget = widgets.Checkbox(
    value=False,
    description="Use other chambers as fallback",
    indent=False,
    layout=wide,
)
donor_chambers_widget = widgets.SelectMultiple(
    options=[],
    value=(),
    description="Permitted donors",
    disabled=True,
    style=style,
    layout=widgets.Layout(width="600px", height="110px"),
)
donor_gap_widget = widgets.FloatText(
    value=30.0,
    description="Maximum donor gap [min]",
    disabled=True,
    style=style,
    layout=medium,
)
auxiliary_status_widget = widgets.HTML(value="Scan dC/dt input first.")
auxiliary_output = widgets.Output()

calculate_button = widgets.Button(
    description="Calculate flux", button_style="info", disabled=True
)
calculation_status_widget = widgets.HTML(value="Load dC/dt and auxiliary inputs first.")
calculation_output = widgets.Output()

preview_file_widget = widgets.SelectMultiple(
    options=[],
    value=(),
    description="Preview results",
    disabled=True,
    style=style,
    layout=widgets.Layout(width="760px", height="120px"),
)
preview_select_all_button = widgets.Button(
    description="Select all previews", disabled=True
)
preview_clear_selection_button = widgets.Button(
    description="Clear preview selection", disabled=True
)
preview_deadband_widget = widgets.Dropdown(
    options=[],
    description="Deadband",
    disabled=True,
    style=style,
    layout=medium,
)
preview_cutoff_widget = widgets.Dropdown(
    options=[],
    description="Cutoff",
    disabled=True,
    style=style,
    layout=medium,
)
_EMPTY_DATE = np.datetime64("1970-01-01T00:00:00")
preview_date_widget = widgets.SelectionRangeSlider(
    options=[("No timestamps loaded", _EMPTY_DATE)],
    value=(_EMPTY_DATE, _EMPTY_DATE),
    description="Timestamp range",
    disabled=True,
    continuous_update=False,
    style=style,
    layout=wide,
)
preview_y_widget = widgets.FloatRangeSlider(
    value=(0.0, 1.0),
    min=0.0,
    max=1.0,
    step=0.01,
    description="Flux y range",
    disabled=True,
    continuous_update=False,
    readout_format=".4g",
    style=style,
    layout=wide,
)
preview_filter_y_widget = widgets.Checkbox(
    value=True,
    description="Filter points outside the y range",
    indent=False,
    disabled=True,
    layout=wide,
)
preview_smooth_widget = widgets.Checkbox(
    value=False,
    description="Moving-window mean",
    indent=False,
    disabled=True,
    layout=medium,
)
preview_window_size_widget = widgets.BoundedIntText(
    value=60, min=1, max=100000,
    description="Window size",
    disabled=True,
    style=style,
    layout=medium,
)
preview_window_unit_widget = widgets.Dropdown(
    options=["minutes", "hours", "days"],
    value="minutes",
    description="Window unit",
    disabled=True,
    style=style,
    layout=medium,
)
preview_status_widget = widgets.HTML(
    value="Calculate flux to enable preview filtering and smoothing."
)
preview_output = widgets.Output()

flux_output_folder_widget = widgets.Textarea(
    value=str(DEFAULT_FLUX_OUTPUT_DIR),
    description="Flux output folder",
    continuous_update=False,
    style=style,
    layout=path_layout,
)
overwrite_flux_widget = widgets.Checkbox(
    value=False,
    description="Overwrite existing files",
    indent=False,
    layout=medium,
)
export_flux_button = widgets.Button(
    description="Export NetCDF files", button_style="success", disabled=True
)
export_output = widgets.Output()


def display_exception(exc):
    print(f"{type(exc).__name__}: {exc}")


def close_calculated():
    for entry in state["calculated"]:
        try:
            entry["dataset"].close()
        except Exception:
            pass
    state["calculated"] = []


def clear_calculation():
    close_calculated()
    calculate_button.disabled = not bool(state["auxiliary_catalog"])
    export_flux_button.disabled = True
    preview_file_widget.options = []
    preview_file_widget.value = ()
    preview_file_widget.disabled = True
    preview_select_all_button.disabled = True
    preview_clear_selection_button.disabled = True
    preview_deadband_widget.options = []
    preview_deadband_widget.disabled = True
    preview_cutoff_widget.options = []
    preview_cutoff_widget.disabled = True
    preview_date_widget.options = [("No timestamps loaded", _EMPTY_DATE)]
    preview_date_widget.value = (_EMPTY_DATE, _EMPTY_DATE)
    preview_date_widget.disabled = True
    preview_y_widget.disabled = True
    preview_filter_y_widget.disabled = True
    preview_smooth_widget.disabled = True
    preview_window_size_widget.disabled = True
    preview_window_unit_widget.disabled = True
    preview_status_widget.value = (
        "Calculate flux to enable preview filtering and smoothing."
    )
    calculation_status_widget.value = (
        "Review auxiliary options, then calculate flux."
        if state["auxiliary_catalog"]
        else "Load dC/dt and auxiliary inputs first."
    )
    with calculation_output:
        clear_output(wait=True)
    with preview_output:
        clear_output(wait=True)
    with export_output:
        clear_output(wait=True)


def clear_auxiliary():
    state["auxiliary_paths"] = []
    state["auxiliary_records"] = []
    state["auxiliary_catalog"] = []
    donor_chambers_widget.options = []
    donor_chambers_widget.value = ()
    scan_auxiliary_button.disabled = not bool(state["result_records"])
    auxiliary_status_widget.value = (
        "Scan the auxiliary folder to match the selected dC/dt measurements."
        if state["result_records"]
        else "Scan dC/dt input first."
    )
    with auxiliary_output:
        clear_output(wait=True)
    clear_calculation()


def clear_results():
    state["result_records"] = []
    state["result_scan_records"] = []
    result_status_widget.value = "Scan the dC/dt input."
    with result_output:
        clear_output(wait=True)
    clear_auxiliary()


def on_result_mode_changed(change=None):
    folder_schema_widget.disabled = result_mode_widget.value != "Whole folder"
    clear_results()


def on_scan_results_clicked(_):
    clear_results()
    with result_output:
        clear_output(wait=True)
        try:
            selected, records = discover_result_inputs(
                result_mode_widget.value,
                result_path_widget.value,
                folder_schema_widget.value,
            )
            state["result_records"] = selected
            state["result_scan_records"] = records
            display(result_records_table(records))
            schemas = sorted({record["schema"] for record in selected})
            result_status_widget.value = (
                f"<b>Selected:</b> {len(selected)} compatible file(s); "
                f"schema(s): {', '.join(schemas)}."
            )
            scan_auxiliary_button.disabled = False
            auxiliary_status_widget.value = (
                "Select and scan the auxiliary folder. Primary files will follow "
                "the selected result chambers and timestamps automatically."
            )
        except Exception as exc:
            display_exception(exc)
            result_status_widget.value = (
                f"<b>Result scan failed:</b> {type(exc).__name__}: {exc}"
            )


def on_scan_auxiliary_clicked(_):
    clear_auxiliary()
    with auxiliary_output:
        clear_output(wait=True)
        try:
            paths, records = discover_auxiliary_files(auxiliary_path_widget.value)
            catalog = load_auxiliary_catalog(paths)
            state["auxiliary_paths"] = paths
            state["auxiliary_records"] = records
            state["auxiliary_catalog"] = catalog

            display(auxiliary_records_table(records))
            summary = summarize_primary_matches(state["result_records"], catalog)
            display(summary)
            counts = summary["match_state"].value_counts()
            chambers = sorted({measurement["chamber_id"] for measurement in catalog})
            state["updating"] = True
            try:
                donor_chambers_widget.options = chambers
                donor_chambers_widget.value = ()
            finally:
                state["updating"] = False

            auxiliary_status_widget.value = (
                f"<b>Loaded:</b> {len(paths)} auxiliary file(s), "
                f"{len(catalog)} measurement(s). Exact primary matches: "
                f"{int(counts.get('matched_valid', 0))} valid, "
                f"{int(counts.get('matched_invalid', 0))} invalid, "
                f"{int(counts.get('ambiguous', 0))} ambiguous, "
                f"{int(counts.get('missing', 0))} missing."
            )
            calculate_button.disabled = False
            calculation_status_widget.value = (
                "Review auxiliary reduction and donor options, then calculate flux."
            )
        except Exception as exc:
            display_exception(exc)
            auxiliary_status_widget.value = (
                f"<b>Auxiliary scan failed:</b> {type(exc).__name__}: {exc}"
            )


def on_use_donor_changed(change=None):
    enabled = bool(use_donor_widget.value)
    donor_chambers_widget.disabled = not enabled
    donor_gap_widget.disabled = not enabled
    if not state["updating"]:
        clear_calculation()


def on_calculation_option_changed(change=None):
    if not state["updating"] and state["auxiliary_catalog"]:
        clear_calculation()


def set_preview_selection(indexes):
    preview_file_widget.value = tuple(int(index) for index in indexes)


def current_preview_entries():
    if not state["calculated"]:
        return []
    indexes = tuple(preview_file_widget.value or ())
    return [
        state["calculated"][int(index)]
        for index in indexes
        if 0 <= int(index) < len(state["calculated"])
    ]


def common_coordinate_values(entries, coordinate):
    if not entries:
        return []
    ordered = [
        value.item() if hasattr(value, "item") else value
        for value in entries[0]["dataset"][coordinate].values
    ]
    common = set(ordered)
    for entry in entries[1:]:
        values = {
            value.item() if hasattr(value, "item") else value
            for value in entry["dataset"][coordinate].values
        }
        common &= values
    return [value for value in ordered if value in common]


def disable_preview_controls(message):
    state["updating"] = True
    try:
        preview_deadband_widget.options = []
        preview_deadband_widget.disabled = True
        preview_cutoff_widget.options = []
        preview_cutoff_widget.disabled = True
        preview_date_widget.options = [("No timestamps loaded", _EMPTY_DATE)]
        preview_date_widget.value = (_EMPTY_DATE, _EMPTY_DATE)
        preview_date_widget.disabled = True
        preview_y_widget.disabled = True
        preview_filter_y_widget.disabled = True
        preview_smooth_widget.disabled = True
        preview_window_size_widget.disabled = True
        preview_window_unit_widget.disabled = True
    finally:
        state["updating"] = False
    preview_status_widget.value = message
    with preview_output:
        clear_output(wait=True)


def configure_preview_for_entries(change=None):
    if state["updating"]:
        return
    entries = current_preview_entries()
    if not entries:
        disable_preview_controls(
            "Select one or more calculated results to preview."
        )
        return
    schemas = {entry["schema"] for entry in entries}
    if len(schemas) != 1:
        disable_preview_controls(
            "Selected previews must all use the same result schema."
        )
        return

    schema = entries[0]["schema"]
    state["updating"] = True
    try:
        if schema == STANDARD_SCHEMA:
            deadband_options = common_coordinate_values(entries, "deadband")
            cutoff_options = common_coordinate_values(entries, "cutoff")
            if not deadband_options or not cutoff_options:
                raise ValueError(
                    "Selected Standard files share no deadband/cutoff coordinates."
                )
            old_deadband = preview_deadband_widget.value
            old_cutoff = preview_cutoff_widget.value
            preview_deadband_widget.options = deadband_options
            preview_deadband_widget.value = (
                old_deadband if old_deadband in deadband_options else deadband_options[0]
            )
            preview_cutoff_widget.options = cutoff_options
            preview_cutoff_widget.value = (
                old_cutoff if old_cutoff in cutoff_options else cutoff_options[0]
            )
            preview_deadband_widget.disabled = False
            preview_cutoff_widget.disabled = False
        else:
            preview_deadband_widget.options = []
            preview_cutoff_widget.options = []
            preview_deadband_widget.disabled = True
            preview_cutoff_widget.disabled = True

        all_times = np.concatenate([
            np.asarray(entry["dataset"]["time"].values).reshape(-1)
            for entry in entries
        ])
        options = timestamp_options(all_times)
        preview_date_widget.options = options
        preview_date_widget.value = (options[0][1], options[-1][1])
        preview_date_widget.disabled = False
    except Exception as exc:
        state["updating"] = False
        disable_preview_controls(f"<b>Preview unavailable:</b> {exc}")
        return
    finally:
        state["updating"] = False
    configure_preview_y_range()


def configure_preview_y_range(change=None):
    if state["updating"]:
        return
    entries = current_preview_entries()
    if not entries:
        return
    try:
        value_parts = []
        for entry in entries:
            view = selected_flux_view(
                entry,
                deadband=preview_deadband_widget.value,
                cutoff=preview_cutoff_widget.value,
            )
            values = (
                view["flux"].values
                if entry["schema"] == STANDARD_SCHEMA
                else view[["q16", "median", "q84"]].values
            )
            value_parts.append(np.asarray(values, dtype=float).reshape(-1))
        lower, upper, step = finite_bounds(np.concatenate(value_parts))
        state["updating"] = True
        try:
            # Expand bounds first so positive-only or negative-only ranges
            # cannot violate the slider's current trait constraints.
            preview_y_widget.min = min(preview_y_widget.min, lower)
            preview_y_widget.max = max(preview_y_widget.max, upper)
            preview_y_widget.step = step
            preview_y_widget.value = (lower, upper)
            preview_y_widget.min = lower
            preview_y_widget.max = upper
            preview_y_widget.disabled = False
            preview_filter_y_widget.disabled = False
            preview_smooth_widget.disabled = False
            preview_window_size_widget.disabled = not preview_smooth_widget.value
            preview_window_unit_widget.disabled = not preview_smooth_widget.value
        finally:
            state["updating"] = False
        render_preview()
    except Exception as exc:
        preview_y_widget.disabled = True
        preview_filter_y_widget.disabled = True
        preview_smooth_widget.disabled = True
        preview_window_size_widget.disabled = True
        preview_window_unit_widget.disabled = True
        preview_status_widget.value = f"<b>Preview unavailable:</b> {exc}"
        with preview_output:
            clear_output(wait=True)
            display_exception(exc)


def on_preview_smoothing_changed(change=None):
    enabled = bool(preview_smooth_widget.value) and not preview_smooth_widget.disabled
    preview_window_size_widget.disabled = not enabled
    preview_window_unit_widget.disabled = not enabled
    render_preview()


def render_preview(change=None):
    if state["updating"]:
        return
    entries = current_preview_entries()
    if not entries:
        return
    with preview_output:
        clear_output(wait=True)
        try:
            fig = plot_flux_entries(
                entries,
                preview_deadband_widget.value,
                preview_cutoff_widget.value,
                preview_date_widget.value,
                preview_y_widget.value if not preview_y_widget.disabled else None,
                filter_y=preview_filter_y_widget.value,
                smooth=preview_smooth_widget.value,
                window_size=preview_window_size_widget.value,
                window_unit=preview_window_unit_widget.value,
            )
            display(fig)
            plt.close(fig)
            raw_view_count = 0
            filtered_count = 0
            provenance_parts = []
            for entry in entries:
                raw_view = selected_flux_view(
                    entry,
                    deadband=preview_deadband_widget.value,
                    cutoff=preview_cutoff_widget.value,
                )
                if preview_date_widget.value:
                    preview_start, preview_end = [
                        utc_naive_timestamp(value) for value in preview_date_widget.value
                    ]
                    raw_view = raw_view.loc[
                        (raw_view.index >= preview_start)
                        & (raw_view.index <= preview_end)
                    ]
                raw_view_count += len(raw_view)
                if preview_filter_y_widget.value:
                    filtered_count += count_flux_outside_y_range(
                        raw_view, entry["schema"], preview_y_widget.value
                    )

                provenance = provenance_table(
                    entry,
                    deadband=preview_deadband_widget.value,
                    cutoff=preview_cutoff_widget.value,
                )
                provenance.insert(0, "target_chamber", entry["chamber_id"])
                provenance.insert(0, "result_file", entry["file"])
                if preview_date_widget.value:
                    provenance = provenance[
                        provenance["dcdt_time"].between(
                            preview_start, preview_end, inclusive="both"
                        )
                    ]
                provenance_parts.append(provenance)
            filter_note = (
                f" Filtered {filtered_count} raw point(s) outside the y range."
                if filtered_count else ""
            )
            smooth_note = (
                f" Applied a centered {preview_window_size_widget.value} "
                f"{preview_window_unit_widget.value} moving-window mean."
                if preview_smooth_widget.value else ""
            )
            preview_status_widget.value = (
                f"<b>Preview:</b> {len(entries)} file(s), "
                f"{raw_view_count} file-timestamp point(s).{filter_note}"
                f"{smooth_note} Filtering and smoothing affect this plot only."
            )
            display(pd.concat(provenance_parts, ignore_index=True))
        except Exception as exc:
            preview_status_widget.value = f"<b>Preview failed:</b> {type(exc).__name__}: {exc}"
            display_exception(exc)


def on_calculate_clicked(_):
    clear_calculation()
    with calculation_output:
        clear_output(wait=True)
        try:
            calculated, errors = calculate_flux_batch(
                state["result_records"],
                state["auxiliary_catalog"],
                state["auxiliary_paths"],
                reduction_widget.value,
                use_other_chambers=use_donor_widget.value,
                permitted_donor_chambers=donor_chambers_widget.value,
                maximum_gap_minutes=float(donor_gap_widget.value),
            )
            state["calculated"] = calculated
            if calculated:
                counts = flux_result_counts(calculated)
                display(counts)
                state["updating"] = True
                try:
                    preview_file_widget.options = [
                        (f"{entry['file']} — {entry['chamber_id']}", index)
                        for index, entry in enumerate(calculated)
                    ]
                    set_preview_selection([0])
                    preview_file_widget.disabled = False
                    preview_select_all_button.disabled = False
                    preview_clear_selection_button.disabled = False
                finally:
                    state["updating"] = False
                export_flux_button.disabled = False
                calculation_status_widget.value = (
                    f"<b>Calculated:</b> {len(calculated)} file(s); "
                    f"{len(errors)} file(s) failed."
                )
                configure_preview_for_entries()
            else:
                calculation_status_widget.value = "<b>No files were calculated.</b>"
            if errors:
                print("Calculation errors:")
                display(pd.DataFrame(errors))
        except Exception as exc:
            display_exception(exc)
            calculation_status_widget.value = (
                f"<b>Calculation failed:</b> {type(exc).__name__}: {exc}"
            )


def on_export_clicked(_):
    with export_output:
        clear_output(wait=True)
        try:
            written = export_flux_entries(
                state["calculated"],
                flux_output_folder_widget.value,
                overwrite=overwrite_flux_widget.value,
            )
            print(f"Wrote {len(written)} NetCDF file(s):")
            for path in written:
                print(f"  {path}")
        except Exception as exc:
            display_exception(exc)


def on_preview_select_all_clicked(_):
    set_preview_selection(range(len(state["calculated"])))


def on_preview_clear_selection_clicked(_):
    set_preview_selection([])


result_mode_widget.observe(on_result_mode_changed, names="value")
result_path_widget.observe(lambda change: clear_results(), names="value")
folder_schema_widget.observe(lambda change: clear_results(), names="value")
scan_results_button.on_click(on_scan_results_clicked)
auxiliary_path_widget.observe(lambda change: clear_auxiliary(), names="value")
scan_auxiliary_button.on_click(on_scan_auxiliary_clicked)
use_donor_widget.observe(on_use_donor_changed, names="value")
reduction_widget.observe(on_calculation_option_changed, names="value")
donor_chambers_widget.observe(on_calculation_option_changed, names="value")
donor_gap_widget.observe(on_calculation_option_changed, names="value")
calculate_button.on_click(on_calculate_clicked)
preview_file_widget.observe(configure_preview_for_entries, names="value")
preview_select_all_button.on_click(on_preview_select_all_clicked)
preview_clear_selection_button.on_click(on_preview_clear_selection_clicked)
preview_deadband_widget.observe(configure_preview_y_range, names="value")
preview_cutoff_widget.observe(configure_preview_y_range, names="value")
preview_date_widget.observe(render_preview, names="value")
preview_y_widget.observe(render_preview, names="value")
preview_filter_y_widget.observe(render_preview, names="value")
preview_smooth_widget.observe(on_preview_smoothing_changed, names="value")
preview_window_size_widget.observe(render_preview, names="value")
preview_window_unit_widget.observe(render_preview, names="value")
export_flux_button.on_click(on_export_clicked)

results_tab = widgets.VBox([
    widgets.HTML("<h3>dC/dt result input</h3>"),
    result_mode_widget,
    result_path_widget,
    folder_schema_widget,
    scan_results_button,
    result_status_widget,
    result_output,
])

auxiliary_tab = widgets.VBox([
    widgets.HTML("<h3>Auxiliary input and optional donor chambers</h3>"),
    auxiliary_path_widget,
    scan_auxiliary_button,
    reduction_widget,
    use_donor_widget,
    donor_chambers_widget,
    donor_gap_widget,
    auxiliary_status_widget,
    auxiliary_output,
])

flux_tab = widgets.VBox([
    widgets.HTML("<h3>Calculate, preview, and export flux</h3>"),
    calculate_button,
    calculation_status_widget,
    calculation_output,
    preview_file_widget,
    widgets.HBox([preview_select_all_button, preview_clear_selection_button]),
    widgets.HBox([preview_deadband_widget, preview_cutoff_widget]),
    preview_date_widget,
    preview_y_widget,
    preview_filter_y_widget,
    widgets.HBox([
        preview_smooth_widget,
        preview_window_size_widget,
        preview_window_unit_widget,
    ]),
    preview_status_widget,
    preview_output,
    flux_output_folder_widget,
    overwrite_flux_widget,
    export_flux_button,
    export_output,
])

tabs = widgets.Tab(children=[results_tab, auxiliary_tab, flux_tab])
tabs.set_title(0, "dC/dt Input")
tabs.set_title(1, "Auxiliary Input")
tabs.set_title(2, "Flux & Export")
display(tabs)
